[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/Datascience-Practice-Notebooks/blob/main/01_Linear_Regression_California_Housing.ipynb)

# Linear Regression — California Housing Prices

**Problem type:** supervised regression (predict a continuous number)

---

## What is Linear Regression?

Linear regression is one of the oldest and most interpretable machine-learning
algorithms. It models the relationship between a set of input features
**X** and a continuous target **y** by fitting a straight (hyper)plane:

```
ŷ = w₀ + w₁x₁ + w₂x₂ + … + wₙxₙ
```

The model is trained by minimising the **mean squared error** (MSE) between
the predicted values **ŷ** and the actual values **y**.

**How it differs from classification:**  
- *Classification* predicts a discrete label (e.g. "cat" or "dog", spam or not spam).  
- *Regression* predicts a **continuous number** (e.g. a house price, a temperature,
  a salary). The loss function, evaluation metrics (RMSE, MAE, R²), and the
  decision boundary concept are all different.

---

## Dataset — California Housing (scikit-learn)

| Property | Value |
|---|---|
| Source | 1990 US Census, California block groups |
| Rows | 20,640 districts |
| Features | 8 numeric features |
| Target | Median house value (in \$100,000s) |

**Features:**  
`MedInc` — median income (in tens of thousands \$)  
`HouseAge` — median house age  
`AveRooms`, `AveBedrms` — average rooms / bedrooms per household  
`Population` — block-group population  
`AveOccup` — average household occupancy  
`Latitude`, `Longitude` — geographic co-ordinates  


In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # non-interactive backend (works in Colab too)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
print('Libraries loaded successfully.')


## 1. Load the Dataset


In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame                 # combined DataFrame (features + target)
X_raw = housing.data
y_raw = housing.target             # median house value in $100k units

print('Shape:', df.shape)
df.head()


In [ ]:
df.describe().round(2)


## 2. Exploratory Data Analysis


In [ ]:
# ── 2a. Target distribution ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(y_raw, bins=50, color='steelblue', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Median House Value ($100k)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Median House Values', fontsize=14)
ax.axvline(y_raw.mean(), color='tomato', linestyle='--', label=f'Mean = {y_raw.mean():.2f}')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/_eda_target.png', dpi=80)
plt.show()
print(f'Target — mean: {y_raw.mean():.3f}  std: {y_raw.std():.3f}  '
      f'min: {y_raw.min():.3f}  max: {y_raw.max():.3f}')


In [ ]:
# ── 2b. Feature correlation with target ──────────────────────────────────────
corr = df.corr()['MedHouseVal'].drop('MedHouseVal').sort_values()
colors = ['tomato' if c < 0 else 'steelblue' for c in corr]
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with House Value', fontsize=12)
ax.set_title('Feature Correlation with Target', fontsize=14)
plt.tight_layout()
plt.savefig('/tmp/_eda_corr.png', dpi=80)
plt.show()
print('Strongest positive predictor:', corr.idxmax(), f'({corr.max():.3f})')
print('Strongest negative predictor:', corr.idxmin(), f'({corr.min():.3f})')
print()
print('Note: MedInc (median income) is the strongest positive predictor of house value.')
print('Latitude has a strong negative correlation — more northern districts tend to be cheaper.')


In [ ]:
# ── 2c. Geographic scatter — house value across California ───────────────────
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(
    df['Longitude'], df['Latitude'],
    c=df['MedHouseVal'], cmap='coolwarm',
    s=0.8, alpha=0.5
)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('Median House Value ($100k)', fontsize=10)
ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude', fontsize=11)
ax.set_title('California House Values by Location\n'
             '(red = expensive, blue = cheap)', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/_eda_geo.png', dpi=80)
plt.show()
print('Expensive clusters visible along the coast: San Francisco Bay Area & Los Angeles.')


## 3. Preprocessing

We split the data 80 / 20 (train / test) and apply `StandardScaler` inside
a `Pipeline` for the linear models. Tree-based models do **not** require scaling
because they split on thresholds, not distances.


In [ ]:
X = housing.data
y = housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set : {X_train.shape[0]:,} rows')
print(f'Test set     : {X_test.shape[0]:,} rows')
print(f'Features     : {X_train.shape[1]}')


## 4. Train Models

We compare four models ranging from simple to complex:

| Model | Key idea |
|---|---|
| **Linear Regression** | Fits a hyperplane; fast and interpretable |
| **Ridge** | Linear + L2 regularisation to reduce overfitting |
| **Random Forest** | Ensemble of 200 decision trees (bagging) |
| **Gradient Boosting** | Ensemble that adds trees to fix previous errors |


In [ ]:
# Build models — linear ones use a scaling pipeline
models = {
    'LinearRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LinearRegression())
    ]),
    'Ridge': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  Ridge(alpha=1.0))
    ]),
    'RandomForest': RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=200, random_state=42
    ),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f'{name:<22}  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}')


## 5. Evaluate & Compare


In [ ]:
# ── 5a. Comparison table ────────────────────────────────────────────────────
results_df = pd.DataFrame(results).T.round(4)
results_df = results_df.sort_values('RMSE')
print(results_df.to_string())


In [ ]:
# ── 5b. 5-fold cross-validation on the best model ───────────────────────────
best_name = results_df.index[0]          # lowest RMSE
best_model = models[best_name]

cv_scores = cross_val_score(
    best_model, X, y,
    cv=5, scoring='r2', n_jobs=-1
)
print(f'Best model : {best_name}')
print(f'5-fold CV R² scores : {np.round(cv_scores, 4)}')
print(f'Mean CV R²  = {cv_scores.mean():.4f}  ±  {cv_scores.std():.4f}')


In [ ]:
# ── 5c. Predicted vs Actual scatter ─────────────────────────────────────────
y_pred_best = models[best_name].predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
ax = axes[0]
ax.scatter(y_test, y_pred_best, alpha=0.25, s=5, color='steelblue')
lo, hi = y_test.min(), y_test.max()
ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.5, label='Perfect fit')
ax.set_xlabel('Actual Value ($100k)', fontsize=11)
ax.set_ylabel('Predicted Value ($100k)', fontsize=11)
ax.set_title(f'{best_name}: Predicted vs Actual', fontsize=13)
ax.legend()

# Residuals
residuals = y_test - y_pred_best
ax = axes[1]
ax.hist(residuals, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='tomato', linestyle='--', linewidth=1.5)
ax.set_xlabel('Residual (Actual − Predicted)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Residual Distribution', fontsize=13)

plt.suptitle(f'Model: {best_name}', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/_eval_scatter.png', dpi=80)
plt.show()
print(f'Residuals — mean: {residuals.mean():.4f}  std: {residuals.std():.4f}')


In [ ]:
# ── 5d. Feature importances (RandomForest) ───────────────────────────────────
rf = models['RandomForest']
importances = pd.Series(
    rf.feature_importances_,
    index=housing.feature_names
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(importances.index, importances.values, color='steelblue')
ax.set_xlabel('Feature Importance (mean decrease in impurity)', fontsize=11)
ax.set_title('RandomForest — Feature Importances', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/_feat_imp.png', dpi=80)
plt.show()

top = importances.idxmax()
print(f'Top feature: {top}  ({importances.max():.4f})')
print('MedInc (median income) strongly dominates — location features come next.')


## Findings

| Model | RMSE | MAE | R² |
|---|---|---|---|
| Linear Regression | ~0.746 | ~0.533 | ~0.576 |
| Ridge | ~0.746 | ~0.533 | ~0.576 |
| Random Forest | ~0.504 | ~0.328 | ~0.806 |
| Gradient Boosting | ~0.536 | ~0.372 | ~0.776 |

**Plain-English summary:**

1. **Linear models** (plain and Ridge) explain roughly **58% of the variance**
   in house prices (R² ≈ 0.576). An RMSE of ~\$74,600 means the average
   prediction is off by about \$75k — not bad for a straight line, but the
   relationship is clearly non-linear.

2. **Ensemble methods** push R² up to **~0.81 (RandomForest)** and ~0.78
   (GradientBoosting), cutting RMSE by ~33% compared with linear regression.
   They capture interaction effects and non-linearities that linear models miss.

3. **Income dominates.** The single most important predictor is `MedInc`
   (median income of the block group) — both in correlation analysis and
   RandomForest feature importance. Geographic features (`Latitude`,
   `Longitude`) come next, reflecting California's well-known coastal premium.

4. **Ridge regularisation** gives virtually identical results to plain OLS here
   because the dataset is large and features are not collinear.

5. **Residuals** for the best model are roughly centred at zero and
   approximately normal, suggesting no major systematic bias — though the
   hard cap at \$500,001 in the original census data creates a visible spike
   at the right tail.


## Try It Yourself

Here are three experiments to deepen your understanding:

### Experiment 1 — Polynomial features
Add `sklearn.preprocessing.PolynomialFeatures(degree=2)` before the scaler in
the Linear Regression pipeline.  
**Prediction:** R² should climb towards 0.65–0.70. Why? Quadratic terms
capture some of the curvature in the income–price relationship.

```python
from sklearn.preprocessing import PolynomialFeatures
poly_lr = Pipeline([
    ('poly',   PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model',  LinearRegression()),
])
poly_lr.fit(X_train, y_train)
y_pred_poly = poly_lr.predict(X_test)
print('Poly R²:', r2_score(y_test, y_pred_poly).round(4))
```

### Experiment 2 — Tune Random Forest depth
Add `max_depth=5` to `RandomForestRegressor` and compare RMSE.  
**Prediction:** performance should drop (more bias) — demonstrating that
deep trees are needed to capture the complexity of this dataset.

```python
rf_shallow = RandomForestRegressor(
    n_estimators=200, max_depth=5, random_state=42, n_jobs=-1
)
rf_shallow.fit(X_train, y_train)
print('Shallow RF R²:', r2_score(y_test, rf_shallow.predict(X_test)).round(4))
```

### Experiment 3 — Drop geographic features
Remove `Latitude` and `Longitude` from the feature matrix and re-train
the RandomForest.  
**Prediction:** R² should fall by 5–10 points, confirming that *where* a
house is located matters almost as much as *who lives there*.

```python
geo_cols = ['Latitude', 'Longitude']
X_no_geo = X.drop(columns=geo_cols)
Xtr_ng, Xte_ng, ytr, yte = train_test_split(
    X_no_geo, y, test_size=0.2, random_state=42
)
rf_no_geo = RandomForestRegressor(
    n_estimators=200, random_state=42, n_jobs=-1
)
rf_no_geo.fit(Xtr_ng, ytr)
print('No-geography RF R²:', r2_score(yte, rf_no_geo.predict(Xte_ng)).round(4))
```
